#### Auto Loader is a Databricks feature that incrementally discovers and processes new files arriving in cloud storage, such as Azure Data Lake Storage Gen2 (ADLS Gen2), and loads their data into a Delta table.
It is commonly used to build scalable ETL pipelines that continuously ingest files without having to manually list and process every file.



“*Auto Loader* is a Databricks feature used for incremental ingestion of files from cloud storage such as ADLS Gen2. It uses the cloudFiles source in Structured Streaming to discover new files, track ingestion progress through checkpointing, and manage schema inference and evolution. In my ETL pipelines, I use Auto Loader to ingest source files into Bronze Delta tables, followed by transformations into Silver and Gold layers. For production workloads, I configure schema handling, persistent checkpoints, monitoring, and Lakeflow Jobs for scheduling and recovery.”

**Remember these four keywords: cloudFiles, schemaLocation, checkpointLocation, and schema evolution.**

![image_1792316662311.png](./image_1792316662311.png "image_1792316662311.png")

#### 1. cloudFiles — Enables Auto Loader
Meaning: cloudFiles is the source format that tells Databricks to use Auto Loader for incremental file ingestion.

In [0]:
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .load("/mnt/raw/orders/")
)

Here:
- format("cloudFiles") enables Auto Loader.
- cloudFiles.format = json specifies that the incoming files are JSON.
- load() specifies the source directory.

### 2. schemaLocation — Stores schema information
Meaning: It specifies where Auto Loader stores schema information, including the schema inferred from incoming files and updates related to schema evolution.

In [0]:
.option(
    "cloudFiles.schemaLocation",
    "/Volumes/financepnd/bronze/metadata/orders_schema"
)

Suppose the original JSON contains:

{"order_id": 101, "amount": 500}

Later new file contains


{"order_id": 102, "amount": 800, "city": "Pune"}

Auto Loader can detect the new city column, depending on the configured schema evolution mode.

**Interview answer:** “schemaLocation stores Auto Loader's schema metadata and supports schema inference and schema evolution across ingestion runs.”

#### 3. checkpointLocation — Tracks processing progress
Meaning: It stores Structured Streaming checkpoint information so that the query can recover its progress and state after a failure.

In [0]:
.option(
    "checkpointLocation",
    "/Volumes/main/bronze/metadata/orders_checkpoint"
)

Example:
- First run processes files A, B and C.
- The streaming query records its progress.
- File D arrives later.
- The query restarts with the same checkpoint and discovers newly available files without normally reprocessing previously tracked files.

**Interview answer:** “checkpointLocation maintains streaming progress and state, allowing Auto Loader to recover after failures and track which files have been processed.”

#### 4. Schema evolution — Handles changes in incoming data
Meaning: Schema evolution allows Auto Loader to handle changes in the structure of incoming files, such as new columns.

In [0]:
source_path = "/mnt/raw/orders/"
schema_path = "/mnt/metadata/orders_schema"
checkpoint_path = "/mnt/metadata/orders_checkpoint"

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(source_path)
)

query = (
    df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("main.bronze.orders")
)

query.awaitTermination()

### 1. Why do we need Auto Loader?
Suppose you work on a data engineering project where an application generates order files every day and stores them in ADLS Gen2.
![image_1792315283968.png](./image_1792315283968.png "image_1792315283968.png")

##2. How Auto Loader works
The main components are:
- Source: Cloud storage, such as ADLS Gen2.
- cloudFiles: The Auto Loader source format used in Structured Streaming.
- Schema management: Infers or tracks the structure of incoming data.
- Checkpoint: Tracks streaming progress and helps recover after failures.
- Target: Usually a Delta table in the Bronze layer.
The typical flow is:
1. Files arrive in cloud storage.
2. Auto Loader discovers newly arrived files.
3. Spark reads and processes the file contents.
4. The streaming checkpoint records progress.
5. Records are written to the destination Delta table.
6. The running stream processes additional files as they arrive.

In [0]:
source_path = "/Volumes/financepnd/bronze/landing_json/input_data/"

schema_path = "/Volumes/financepnd/bronze/landing_json/metadata/orders_schema"

checkpoint_path = "/Volumes/financepnd/bronze/landing_json/metadata/orders_checkpoint"

orders_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(source_path)
)

query = (
    orders_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("financepnd.bronze.orders")
)

query.awaitTermination()

Explain the important lines
![image_1792315933606.png](./image_1792315933606.png "image_1792315933606.png")

In [0]:
%sql
select * from financepnd.bronze.orders;

### 4. What happens when new files arrive?
## First run
Auto Loader discovers orders_01.json and orders_02.json, then loads three records into the Delta table.



## Progress is recorded
Checkpoint metadata records ingestion progress so the stream can recover and track previously processed files.



## Second run
When orders_03.json arrives, the next run processes the newly discovered file without normally reprocessing files already tracked by that checkpoint.